# Housing Regression

A focused tabular regression notebook: load a local sample, build a baseline, compare against a naive benchmark, and inspect where the model misses.

## Setup
This JupyterLite-safe notebook uses the local CSV bundled with the site. Use a larger California Housing export in local Jupyter when you want stable scores.

In [ ]:
%pip install pandas numpy scikit-learn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split

RANDOM_STATE = 42
DATA_PATH = '/cases/datasets/housing_sample.csv'
TARGET = 'median_house_value'

def read_portal_csv(path):
    site_path = path if path.startswith('/') else f'/{path}'
    try:
        open_url = __import__('pyodide.http', fromlist=['open_url']).open_url
        return pd.read_csv(open_url(site_path))
    except Exception:
        relative = site_path.lstrip('/')
        candidates = [Path(relative), Path('..') / relative, Path.cwd() / relative, Path.cwd().parent / relative]
        for candidate in candidates:
            if candidate.exists():
                return pd.read_csv(candidate)
        raise FileNotFoundError(f'Could not find {site_path}')

housing = read_portal_csv(DATA_PATH)
feature_cols = [column for column in housing.columns if column != TARGET]
print('Rows:', len(housing))
print('Target:', TARGET)
display(housing.describe().T)

In [ ]:
X = housing[feature_cols].copy()
y = housing[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
)

model = RandomForestRegressor(
    n_estimators=160,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
baseline = np.repeat(y_train.mean(), len(y_test))

rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
baseline_rmse = float(np.sqrt(mean_squared_error(y_test, baseline)))
mae = float(mean_absolute_error(y_test, pred))
baseline_mae = float(mean_absolute_error(y_test, baseline))
r2 = float(r2_score(y_test, pred))

cv = KFold(n_splits=min(3, len(X_train)), shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='neg_mean_absolute_error')
importance = permutation_importance(model, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE)
importance_table = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': importance.importances_mean,
    'importance_std': importance.importances_std,
}).sort_values('importance_mean', ascending=False)
error_table = pd.DataFrame({
    'actual': y_test,
    'predicted': pred,
    'baseline': baseline,
    'absolute_error': np.abs(y_test - pred),
}).sort_values('absolute_error', ascending=False)

print('Model MAE:', round(mae, 4), '| Baseline MAE:', round(baseline_mae, 4))
print('Model RMSE:', round(rmse, 4), '| Baseline RMSE:', round(baseline_rmse, 4))
print('R2:', round(r2, 4))
print('CV MAE mean:', round(float(-cv_scores.mean()), 4), '| spread:', round(float(cv_scores.std()), 4))
display(error_table)
display(importance_table)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--', color='gray')
plt.title('Actual vs predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.grid(True, alpha=0.3)
plt.show()